# **Heart Failure Data: Data Classification**

## Objectives

* Build a predictive model that uses patient data (such as age, maximum heart rate and cholesterol) to predict whether a patient has heart disease or not, and measure how well it does.

## Inputs

* Dataset file data/processed/heart_clean.csv

## Outputs

* the model’s evaluation results (accuracy, confusion matrix, recall and precision).

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Code Institute - Data Analytics\\heart-failure-analysis\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Code Institute - Data Analytics\\heart-failure-analysis'

# Setup

In [4]:
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("data/processed/heart_clean.csv")
df.shape

(918, 13)

In [9]:
X = df.drop(columns="HeartDisease")
y = df["HeartDisease"]

The features, X, are from the original DataFrame minus the HeartDisease column. The target, y, contains the original HeartDisease column and nothing else.  This separation means original data can be processed using the features without the target values being used as input to the process.  The learning stage of the classification process does, however, have access to the target as part of building the model.

In [10]:
X.shape

(918, 12)

In [11]:
y.shape

(918,)

In [12]:
"HeartDisease" in X.columns

False

### Statistical analysis terms

 Term | Meaning |
| :--- | :----- |
| Feature | An input column the model uses for prediction. |
| Target  | The column to be predicted, HeartDisease in this instance. |

N.B. X holds the features and y holds the target.

### One-hot encoding

The model works with numbers and cannot understand word categories, so words are replaced with numbers.  The number scheme must avoid implying that the numbers represent an ordering of values for the model.  To do that a new column is introduced into the data for each word category, except that one column per category is dropped.  The dropped column is redundant as it adds no additional information.  The presence of a word category is indicated with a 1 in its column, and its absence with a 0.  The final column for each category is not needed as that category can be inferred by the presence of zeros in all other category columns.

In [13]:
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)
"HeartDisease" in X_encoded.columns

False

In [16]:
X_encoded.shape

(918, 16)

In [17]:
X_encoded.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Cholesterol_Missing,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,1,1,0,0,1,0,0,0,1
1,49,160,180,0,156,1.0,0,0,0,1,0,1,0,0,1,0
2,37,130,283,0,98,0.0,0,1,1,0,0,0,1,0,0,1
3,48,138,214,0,108,1.5,0,0,0,0,0,1,0,1,1,0
4,54,150,195,0,122,0.0,0,1,0,1,0,1,0,0,0,1


### Splitting the sample into train and test sets

Splitting the patients into two groups gives a group of patients that can be used to train the model (the training set) and a further group that can be used to test the finished model (the test set).  The split was stratified to keep the same proportions of patients with heart disease and without in each set.  The purpose of the split is to hold back a sample of patients from the training phase, so that the model may then be tested on patients it has not seen.  This makes for a fair test.

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

In [19]:
X_train.shape

(734, 16)

In [20]:
X_test.shape

(184, 16)

In [21]:
y_train.shape

(734,)

In [22]:
y_test.shape

(184,)

### Statistical analysis terms

| Term | Meaning |
| :--- | :------ |
| Model | The scoring system, learned from the data, that is used to make predictions. |
| Fit (train) | The model learns by comparing the training patients (X_train) with the known answers (y_train).  The model learns how much weight to assign to each feature. |
| Predict | Use the learned scoring system to arrive at a prediction for each patient. |
| Logistic regression | A simple model that turns a final score into a probability, and then a prediction. |

The model is trained only on the training set to keep the test set in reserve for an unbiased test of the model.  A model tested on patients it had already seen would give an over-optimistic result.

In [30]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [31]:
y_pred.shape

(184,)

In [32]:
y_pred[:10]

array([1, 0, 1, 1, 0, 0, 0, 1, 0, 1], dtype=int64)

In [33]:
y_test.values[:10]

array([1, 1, 1, 1, 0, 0, 0, 1, 0, 1], dtype=int64)

### Model evaluation terms

Consider the operation of a smoke alarm.

| Situation | Alarm rings? | Fire? | Terminology |
| :-------- | :----------- | :---- | :---------- |
| Alarm rings, there is a fire | Yes | Yes | True positive |
| No alarm, no fire | No | No | True negative |
| Alarm rings, no fire | Yes | No | False positive (a false alarm) |
| No alarm, but there is a fire | No | Yes | False negative (a missed fire) |

In our situation, 'fire' represents a patient with heart disease, and 'alarm' means the model predicted it.

| Term | Meaning |
| :--- | :------ |
| Accuracy | The share of all cases correctly identified [(true positives + true negatives)/total cases] |
| Confusion matrix | A small table counting the four alarm situations above. |
| Recall | The proportion of patients who really had heart disease who were flagged by the model [true positives/(true positives + false negatives)] |
| Precision | The proportion of patients flagged by the model who really had heart disease [true positives/(true positives + false positives)] |

In [35]:
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score 

In [38]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.8913043478260869

In [40]:
confusion = confusion_matrix(y_test, y_pred)
confusion

array([[70, 12],
       [ 8, 94]], dtype=int64)

In [42]:
precision = precision_score(y_test, y_pred)
precision

0.8867924528301887

In [43]:
recall = recall_score(y_test, y_pred)
recall

0.9215686274509803

### Reading the Confusion Matrix

| Disease State | Predicted: No Disease | Predicted: Disease |
| :-----------: | :-------------------: | :----------------: |
| Actual: No Disease | 70 true negatives | 12 false positives  |
| Actual: Disease | 8 false negatives | 94 true positives |

### Data Questions and Answers

| Question | Answer |
| :------- | -----: |
| How many patients with heart disease were missed? | 8 out of 102 |
| How many false alarms? | 12 patients without heart disease flagged |
| Of the patients who had heart disease, the proportion caught? | 92% (recall, or sensitivity) |
| Of the patients flagged, the proportion who really had disease? | 89% (precision) |

 In a screening setting like this, recall matters most, because a missed case is the costlier error.  False positives are of less significance than the need to identify as many cases of heart disease as possible.  Minimising false negatives will reduce harm from mistakenly being given a clean bill of health, with the potential extra suffering that that entails.  An emphasis on minimising false positives would not have the same potential for harm reduction.
 
 There should be a discussion with a patient before they consent to being the subject of modelling.  For instance, they should understand that they might have heart disease but the model could pass them as free of heart disease.  In this test set, about 8% of patients who had heart disease were missed.

 False alarms are also not without consequence in a clinical setting.  Of the patients flagged, the 11% of patients (12 of 106) who, in fact, do not have heart disease may experience anxiety and further tests, some of which carry small risks of their own, such as an angiogram.

 ### Is 89% Accuracy Actually Any Good?

 It is important to know how well the model is performing.  Patients may be incorrectly categorised as having heart disease when they don't and vice versa.  Each situation may harm the patient.  Information on the reliability of predictions is, therefore, clinically important.

 One measure is the accuracy of the model, 89%, meaning the model classified 89% of patients correctly.  In contrast, a baseline model that *always* says heart disease would identify all patients who really have heart disease (8 + 94 = 102), but would have an accuracy of about 55% (102/184).  The low accuracy of the baseline arises because it wrongly flags all 82 patients without disease.  The difference between 55% and 89% indicates that the current model beats the baseline by a wide margin.

 There are some caveats to be made about the current model.  Whilst 89% accuracy was established for the current split, a different split of the data might arrive at a different result.  Also, this model has been developed as a learning tool based on a public dataset, and is not intended for diagnosing heart disease in practice.  The age histogram suggests the results mainly apply to a particular demographic of middle-aged and older patients.  How the model might perform with a different demographic is unknown.


---

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [29]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)